# LoFTR: Reproduction of Relative-Pose Evaluation on ScanNet and MegaDepth

**Original implementation:** [zju3dv/LoFTR](https://github.com/zju3dv/LoFTR)  
**Team repository:** [matthewpinto1102-web/LoFTR](https://github.com/matthewpinto1102-web/LoFTR)  
**Evaluation datasets:** ScanNet-1500 (indoor) and MegaDepth-1500 (outdoor)

## Objective and evaluation workflow

This notebook reproduces the relative-camera-pose evaluation of LoFTR using the authors' pretrained indoor and outdoor checkpoints. The evaluation includes **1,500 official image pairs per dataset**. No retraining, fine-tuning, or architectural changes to LoFTR are performed.

Benchmark image pairs are preprocessed and passed through the pretrained LoFTR matcher to obtain correspondence coordinates and confidence scores. The correspondences are used for essential-matrix estimation with RANSAC, recovery of relative rotation and translation direction, and comparison with ground-truth camera poses. Pose errors are summarized by the area under the cumulative recall curve at **5°, 10°, and 20°**. Qualitative visualizations provide illustrative examples of the matching outputs.

## Source code and evaluation resources

The evaluation builds on the team's fork of the original LoFTR repository. The source tree, official test data, and checkpoint files are external dependencies; they are not embedded in the notebook. In a Colab environment, the team repository is located at `/content/LoFTR` (from `https://github.com/matthewpinto1102-web/LoFTR.git`). The notebook's configuration cell defines the repository, benchmark-data, checkpoint, and output locations; environment variables allow these paths to be adjusted.

The pretrained **`indoor_ds_new.ckpt`** checkpoint is evaluated on ScanNet with `temp_bug_fix=True`; **`outdoor_ds.ckpt`** is evaluated on MegaDepth with `temp_bug_fix=False`. The latter setting preserves compatibility with the outdoor checkpoint's positional-encoding implementation. MegaDepth evaluation also retains image-scaling metadata (`scale0`, `scale1`) and padding masks to maintain geometric consistency after resizing.


## 1. Environment and configurable paths

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path
from copy import deepcopy

import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# If using Colab and files are on Google Drive, permit the Drive mount prompt.
MOUNT_GOOGLE_DRIVE = True
if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError:
        pass  # Local Jupyter: no Google Drive integration needed.
    else:
        drive.mount("/content/drive")

# The evaluation code continues from Matthew's team fork of the original LoFTR repository.
TEAM_REPO_URL = "https://github.com/matthewpinto1102-web/LoFTR.git"
# Clone this repository into REPO_ROOT before running (see the instructions above).
# Override these paths using environment variables or edit here.
REPO_ROOT = Path(os.environ.get("LOFTR_REPO_ROOT", "/content/LoFTR")).expanduser().resolve()
RESOURCE_ROOT = Path(os.environ.get("LOFTR_RESOURCE_ROOT", "/content/drive/MyDrive/LoFTR")).expanduser()
RESULTS_DIR = Path(os.environ.get("LOFTR_RESULTS_DIR", "/content/loftr_reproduction_outputs")).expanduser()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not (REPO_ROOT / "src" / "loftr").exists():
    raise FileNotFoundError(
        f"LoFTR repository not found at {REPO_ROOT}. Clone the team's LoFTR fork "
        f"using {TEAM_REPO_URL}, then set LOFTR_REPO_ROOT or edit REPO_ROOT above."
    )
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.loftr import LoFTR, default_cfg
from src.datasets.megadepth import MegaDepthDataset
from src.utils.metrics import estimate_pose as estimate_pose_repo
from src.utils.metrics import relative_pose_error as relative_pose_error_repo

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Repository:", REPO_ROOT)
print("Resources:", RESOURCE_ROOT)
print("Outputs:", RESULTS_DIR)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__, "NumPy:", np.__version__)


In [ ]:
# Paths are grouped in one cell; replace file locations as needed.
SCANNET_ROOT = REPO_ROOT / "data" / "scannet" / "test"
MEGA_ROOT = REPO_ROOT / "data" / "megadepth" / "test"
SCANNET_TEST_NPZ = REPO_ROOT / "assets" / "scannet_test_1500" / "test.npz"
SCANNET_INTR_NPZ = REPO_ROOT / "assets" / "scannet_test_1500" / "intrinsics.npz"
MEGA_SCENE_ROOT = REPO_ROOT / "assets" / "megadepth_test_1500_scene_info"

# Paths include original Google Drive filenames and standard filenames.
def first_existing(*paths):
    options = [Path(p) for p in paths]
    return next((p for p in options if p.exists()), options[0])

SCANNET_TAR = first_existing(
    RESOURCE_ROOT / "testdata" / "Copy of scannet_test_1500.tar",
    RESOURCE_ROOT / "testdata" / "scannet_test_1500.tar",
)
MEGA_TAR = first_existing(
    RESOURCE_ROOT / "testdata" / "Copy of megadepth_test_1500.tar",
    RESOURCE_ROOT / "testdata" / "megadepth_test_1500.tar",
)
INDOOR_CKPT = first_existing(
    RESOURCE_ROOT / "weights" / "Copy of indoor_ds_new.ckpt",
    RESOURCE_ROOT / "weights" / "indoor_ds_new.ckpt",
    REPO_ROOT / "weights" / "indoor_ds_new.ckpt",
)
OUTDOOR_CKPT = first_existing(
    RESOURCE_ROOT / "weights" / "Copy of outdoor_ds.ckpt",
    RESOURCE_ROOT / "weights" / "outdoor_ds.ckpt",
    REPO_ROOT / "weights" / "outdoor_ds.ckpt",
)

# Archive extraction uses the original --strip-components=1 layout. Only run on trusted archives.
AUTO_EXTRACT_IF_MISSING = True

def ensure_dataset(root, marker, archive):
    if (root / marker).exists():
        return
    if not AUTO_EXTRACT_IF_MISSING:
        raise FileNotFoundError(f"Missing dataset at {root}; extract the benchmark archive first.")
    if not archive.is_file():
        raise FileNotFoundError(f"Dataset archive not found: {archive}. Update the paths above.")
    root.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["tar", "-xf", str(archive), "-C", str(root), "--strip-components=1"],
        check=True,
    )
    if not (root / marker).exists():
        raise FileNotFoundError(f"Extraction finished but {marker} was not found under {root}.")

for needed in (SCANNET_TEST_NPZ, SCANNET_INTR_NPZ,
               MEGA_SCENE_ROOT / "megadepth_test_1500.txt", INDOOR_CKPT, OUTDOOR_CKPT):
    if not needed.is_file():
        raise FileNotFoundError(f"Required benchmark metadata or checkpoint not found: {needed}")

ensure_dataset(SCANNET_ROOT, "scene0707_00", SCANNET_TAR)
ensure_dataset(MEGA_ROOT, "Undistorted_SfM", MEGA_TAR)
print("Benchmark files located. Checkpoint weights are used for inference only.")


## 2. ScanNet-1500 · Indoor pretrained inference

In [ ]:
with np.load(SCANNET_TEST_NPZ) as test_npz:
    pairs = test_npz["name"].copy()
with np.load(SCANNET_INTR_NPZ) as intr_npz:
    intr_data = {key: intr_npz[key].copy() for key in intr_npz.files}

assert len(pairs) == 1500, f"Expected 1,500 official ScanNet pairs; found {len(pairs)}"

cfg_indoor = deepcopy(default_cfg)
cfg_indoor["coarse"]["temp_bug_fix"] = True  # updated indoor_ds_new checkpoint
matcher_indoor = LoFTR(config=cfg_indoor)
indoor_weights = torch.load(str(INDOOR_CKPT), map_location="cpu")  # only trusted checkpoint files
matcher_indoor.load_state_dict(indoor_weights["state_dict"])
matcher_indoor = matcher_indoor.eval().to(DEVICE)
del indoor_weights
print("Loaded indoor checkpoint:", INDOOR_CKPT.name)


### 2.1 Qualitative check · one official ScanNet image pair

In [ ]:
scene = "scene0772_00"
frame0, frame1 = 1020, 1050

img0_path = os.path.join(SCANNET_ROOT, scene, "color", f"{frame0}.jpg")
img1_path = os.path.join(SCANNET_ROOT, scene, "color", f"{frame1}.jpg")

img0_raw = cv2.imread(img0_path, cv2.IMREAD_GRAYSCALE)
img1_raw = cv2.imread(img1_path, cv2.IMREAD_GRAYSCALE)

if img0_raw is None or img1_raw is None:
    raise FileNotFoundError(f"Missing example pair: {img0_path}, {img1_path}")

img0_raw = cv2.resize(img0_raw, (640, 480))
img1_raw = cv2.resize(img1_raw, (640, 480))

img0 = torch.from_numpy(img0_raw).float()[None, None].to(DEVICE) / 255.0
img1 = torch.from_numpy(img1_raw).float()[None, None].to(DEVICE) / 255.0

batch = {"image0": img0, "image1": img1}

with torch.no_grad():
    matcher_indoor(batch)

mkpts0 = batch["mkpts0_f"].detach().cpu().numpy()
mkpts1 = batch["mkpts1_f"].detach().cpu().numpy()
mconf = batch["mconf"].detach().cpu().numpy()

print("Number of LoFTR matches:", len(mkpts0))


In [ ]:
# Plot the 150 highest-confidence correspondences
idx = np.argsort(mconf)[::-1][:150]
pts0_vis, pts1_vis = mkpts0[idx], mkpts1[idx]

h = max(img0_raw.shape[0], img1_raw.shape[0])
w0, w1 = img0_raw.shape[1], img1_raw.shape[1]

canvas = np.zeros((h, w0 + w1), dtype=np.uint8)
canvas[:img0_raw.shape[0], :w0] = img0_raw
canvas[:img1_raw.shape[0], w0:w0+w1] = img1_raw

plt.figure(figsize=(16, 8))
plt.imshow(canvas, cmap="gray")

for p0, p1 in zip(pts0_vis, pts1_vis):
    x0, y0 = p0
    x1, y1 = p1
    x1 += w0
    plt.plot([x0, x1], [y0, y1], linewidth=0.7, alpha=0.7)
    plt.scatter(x0, y0, s=5)
    plt.scatter(x1, y1, s=5)

plt.title(f"LoFTR: Top 150 Matches\nTotal predicted matches = {len(mkpts0)}")
plt.axis("off")
plt.show()


### 2.2 Relative-pose estimation and AUC computation

Matched pixel coordinates are normalized using the intrinsic calibration matrices of both cameras. The RANSAC tolerance is converted from pixels to normalized-camera coordinates using the mean of the two cameras' horizontal and vertical focal-length components. An essential matrix is estimated with OpenCV and decomposed to recover the relative rotation and translation **direction**. The angular rotation and translation-direction errors are expressed in **degrees**, with the larger error used as the per-pair pose error. The cumulative recall curve is integrated to obtain AUC@5°, AUC@10°, and AUC@20°.


In [ ]:
# Relative-pose and AUC functions

def estimate_pose_scannet(kpts0, kpts1, K0, K1, pixel_thr=0.5, conf=0.99999):
    if len(kpts0) < 5:
        return None

    # Normalize coordinates by camera intrinsics
    p0 = (kpts0 - K0[[0, 1], [2, 2]]) / K0[[0, 1], [0, 1]]
    p1 = (kpts1 - K1[[0, 1], [2, 2]]) / K1[[0, 1], [0, 1]]

    focal = np.mean([K0[0, 0], K0[1, 1], K1[0, 0], K1[1, 1]])
    ransac_thr = pixel_thr / focal

    E, mask = cv2.findEssentialMat(
        p0, p1, np.eye(3),
        threshold=ransac_thr,
        prob=conf,
        method=cv2.RANSAC
    )

    if E is None or mask is None:
        return None

    best_num_inliers = 0
    best_result = None

    for E_candidate in np.split(E, len(E) // 3):
        n, R, t, _ = cv2.recoverPose(
            E_candidate, p0, p1, np.eye(3), 1e9, mask=mask
        )

        if n > best_num_inliers:
            best_num_inliers = n
            best_result = (R, t[:, 0], mask.ravel() > 0)

    return best_result


def relative_pose_error_scannet(T_gt, R_est, t_est):
    R_gt = T_gt[:3, :3]
    t_gt = T_gt[:3, 3]

    denom = np.linalg.norm(t_est) * np.linalg.norm(t_gt)

    if denom == 0:
        t_err = 0.0
    else:
        cos_t = np.dot(t_est, t_gt) / denom
        cos_t = np.clip(cos_t, -1.0, 1.0)
        t_err = np.degrees(np.arccos(cos_t))
        # Essential-matrix translation has sign ambiguity
        t_err = min(t_err, 180.0 - t_err)

    cos_R = (np.trace(R_est.T @ R_gt) - 1.0) / 2.0
    cos_R = np.clip(cos_R, -1.0, 1.0)
    R_err = np.degrees(np.arccos(cos_R))

    return t_err, R_err


def error_auc(errors, thresholds=(5, 10, 20)):
    errors = [0.0] + sorted(list(errors))
    recall = np.linspace(0, 1, len(errors)).tolist()

    # NumPy 1.x uses trapz; NumPy 2.x also provides trapezoid.
    integrate = getattr(np, "trapezoid", None)
    if integrate is None:
        integrate = np.trapz
    results = {}
    for thr in thresholds:
        last_index = np.searchsorted(errors, thr)
        y = recall[:last_index] + [recall[last_index - 1]]
        x = errors[:last_index] + [thr]
        results[f"AUC@{thr}°"] = integrate(y, x) / thr

    return results


### 2.3 Full official ScanNet-1500 evaluation

In [ ]:
scan_pose_errors = []
scan_rotation_errors = []
scan_translation_errors = []
scan_match_counts = []
scan_inlier_counts = []
scan_failed_pairs = []
scan_failure_reasons = {}

matcher_indoor.eval()
for idx, pair in enumerate(tqdm(pairs, desc="ScanNet-1500")):
    scene_id, subscene_id, frame0, frame1 = map(int, pair)
    scene = f"scene{scene_id:04d}_{subscene_id:02d}"
    pair_pose_error = np.inf
    pair_rot_error = np.inf
    pair_trans_error = np.inf
    pair_matches = 0
    pair_inliers = 0
    failure_reason = None

    try:
        img0_path = SCANNET_ROOT / scene / "color" / f"{frame0}.jpg"
        img1_path = SCANNET_ROOT / scene / "color" / f"{frame1}.jpg"
        pose0_path = SCANNET_ROOT / scene / "pose" / f"{frame0}.txt"
        pose1_path = SCANNET_ROOT / scene / "pose" / f"{frame1}.txt"

        image0 = cv2.imread(str(img0_path), cv2.IMREAD_GRAYSCALE)
        image1 = cv2.imread(str(img1_path), cv2.IMREAD_GRAYSCALE)
        if image0 is None or image1 is None:
            raise FileNotFoundError(f"Missing image: {img0_path} or {img1_path}")
        image0 = cv2.resize(image0, (640, 480))
        image1 = cv2.resize(image1, (640, 480))
        ten0 = torch.from_numpy(image0).float()[None, None].to(DEVICE) / 255.0
        ten1 = torch.from_numpy(image1).float()[None, None].to(DEVICE) / 255.0
        batch = {"image0": ten0, "image1": ten1}
        with torch.no_grad():
            matcher_indoor(batch)
        pts0 = batch["mkpts0_f"].detach().cpu().numpy()
        pts1 = batch["mkpts1_f"].detach().cpu().numpy()
        pair_matches = len(pts0)

        K0 = intr_data[scene].copy()
        K1 = intr_data[scene].copy()
        ret = estimate_pose_scannet(pts0, pts1, K0, K1, pixel_thr=0.5, conf=0.99999)
        if ret is None:
            failure_reason = "pose estimation returned None"
        else:
            R, t, inliers = ret
            pair_inliers = int(inliers.sum())
            cam2world0 = np.loadtxt(pose0_path)
            cam2world1 = np.loadtxt(pose1_path)
            T_gt = np.linalg.inv(cam2world1) @ cam2world0
            pair_trans_error, pair_rot_error = relative_pose_error_scannet(T_gt, R, t)
            pair_pose_error = max(pair_trans_error, pair_rot_error)
    except Exception as exc:
        failure_reason = f"{type(exc).__name__}: {exc}"

    # Append exactly once per official pair, including failures.
    scan_pose_errors.append(float(pair_pose_error))
    scan_rotation_errors.append(float(pair_rot_error))
    scan_translation_errors.append(float(pair_trans_error))
    scan_match_counts.append(int(pair_matches))
    scan_inlier_counts.append(int(pair_inliers))
    if not np.isfinite(pair_pose_error):
        scan_failed_pairs.append(idx)
        scan_failure_reasons[str(idx)] = failure_reason or "non-finite pose error"

assert len(scan_pose_errors) == len(scan_match_counts) == len(scan_inlier_counts) == 1500
print("ScanNet pairs:", len(scan_pose_errors), "failures:", len(scan_failed_pairs))
if scan_failed_pairs:
    print("First failure:", scan_failed_pairs[0], scan_failure_reasons[str(scan_failed_pairs[0])])


In [ ]:
scannet_aucs = error_auc(scan_pose_errors)
finite_scannet = np.asarray(scan_pose_errors, dtype=float)
finite_scannet = finite_scannet[np.isfinite(finite_scannet)]
if len(finite_scannet) == 0:
    raise RuntimeError("ScanNet produced no valid camera poses; check inputs and checkpoint settings.")

scannet_results = {
    "dataset": "ScanNet-1500",
    "checkpoint": INDOOR_CKPT.name,
    "num_pairs": len(scan_pose_errors),
    "failed_pairs": len(scan_failed_pairs),
    "auc_5": float(scannet_aucs["AUC@5°"]),
    "auc_10": float(scannet_aucs["AUC@10°"]),
    "auc_20": float(scannet_aucs["AUC@20°"]),
    "median_pose_error_deg": float(np.median(finite_scannet)),
    "average_matches": float(np.mean(scan_match_counts)),
    "average_ransac_inliers": float(np.mean(scan_inlier_counts)),
}
(RESULTS_DIR / "scannet_reproduction_results.json").write_text(json.dumps(scannet_results, indent=2))
(RESULTS_DIR / "scannet_failure_reasons.json").write_text(json.dumps(scan_failure_reasons, indent=2))
np.save(RESULTS_DIR / "scannet_pose_errors_final.npy", np.asarray(scan_pose_errors, dtype=float))
print("ScanNet AUC (%)", {k: round(v * 100, 2) for k, v in scannet_aucs.items()})
print("ScanNet median pose error (degrees):", scannet_results["median_pose_error_deg"])
print("Saved ScanNet results under", RESULTS_DIR)

# Release GPU memory before the outdoor model is loaded.
del matcher_indoor
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 3. MegaDepth-1500 · Outdoor pretrained inference

In [ ]:
with open(MEGA_SCENE_ROOT / "megadepth_test_1500.txt", encoding="utf-8") as f:
    scenes = [line.strip() for line in f if line.strip()]

cfg_outdoor = deepcopy(default_cfg)
cfg_outdoor["coarse"]["temp_bug_fix"] = False  # compatibility with original outdoor_ds.ckpt
matcher_outdoor = LoFTR(config=cfg_outdoor)
outdoor_weights = torch.load(str(OUTDOOR_CKPT), map_location="cpu")  # only trusted checkpoint files
matcher_outdoor.load_state_dict(outdoor_weights["state_dict"])
matcher_outdoor = matcher_outdoor.eval().to(DEVICE)
del outdoor_weights

# Official MegaDepth test data loader and preprocessing.
datasets = []
for scene_name in scenes:
    ds = MegaDepthDataset(
        root_dir=str(MEGA_ROOT),
        npz_path=str(MEGA_SCENE_ROOT / (scene_name + ".npz")),
        mode="test",
        min_overlap_score=0.0,
        img_resize=840,
        df=8,
        img_padding=True,
        depth_padding=False,
    )
    datasets.append(ds)
    print(scene_name, "pairs:", len(ds))

assert sum(map(len, datasets)) == 1500, "MegaDepth scene list does not total 1,500 pairs."
print("Loaded outdoor checkpoint:", OUTDOOR_CKPT.name)


### 3.1 Qualitative correspondence example

For visual clarity, the plotted example displays the **150 highest-confidence correspondences**. This selection affects only the illustration; quantitative evaluation uses the predicted matches without this visualization limit.


In [ ]:
# Use the first official pair from the first scene subset (order taken from the test scene list).
vis_ds = datasets[0]
vis_sample = vis_ds[0]
vis_batch = {
    "image0": vis_sample["image0"][None].to(DEVICE),
    "image1": vis_sample["image1"][None].to(DEVICE),
    "scale0": vis_sample["scale0"][None].to(DEVICE),
    "scale1": vis_sample["scale1"][None].to(DEVICE),
}
if "mask0" in vis_sample:
    vis_batch["mask0"] = vis_sample["mask0"][None].to(DEVICE)
    vis_batch["mask1"] = vis_sample["mask1"][None].to(DEVICE)
with torch.no_grad():
    matcher_outdoor(vis_batch)

mega_vis_pts0 = vis_batch["mkpts0_f"].detach().cpu().numpy()
mega_vis_pts1 = vis_batch["mkpts1_f"].detach().cpu().numpy()
mega_vis_conf = vis_batch["mconf"].detach().cpu().numpy()
print("MegaDepth example predicted matches:", len(mega_vis_pts0))

# Pair names are supplied by the official MegaDepthDataset rather than hard-coded.
raw0 = cv2.imread(str(MEGA_ROOT / vis_sample["pair_names"][0]))
raw1 = cv2.imread(str(MEGA_ROOT / vis_sample["pair_names"][1]))
if raw0 is None or raw1 is None:
    raise FileNotFoundError("MegaDepth example images missing")
raw0, raw1 = cv2.cvtColor(raw0, cv2.COLOR_BGR2RGB), cv2.cvtColor(raw1, cv2.COLOR_BGR2RGB)
ix = np.argsort(mega_vis_conf)[::-1][:150]
h, w0, w1 = max(raw0.shape[0], raw1.shape[0]), raw0.shape[1], raw1.shape[1]
canvas = np.zeros((h, w0 + w1, 3), dtype=np.uint8)
canvas[:raw0.shape[0], :w0] = raw0
canvas[:raw1.shape[0], w0:w0+w1] = raw1
plt.figure(figsize=(15, 7))
plt.imshow(canvas)
for p0, p1 in zip(mega_vis_pts0[ix], mega_vis_pts1[ix]):
    plt.plot([p0[0], p1[0] + w0], [p0[1], p1[1]], linewidth=0.7, alpha=0.65)
plt.title(f"MegaDepth: top {len(ix)} matches displayed / {len(mega_vis_pts0)} predicted")
plt.axis("off")
plt.tight_layout()
plt.show()


### 3.2 Full official MegaDepth-1500 evaluation

In [ ]:
mega_pose_errors = []
mega_rotation_errors = []
mega_translation_errors = []
mega_match_counts = []
mega_inlier_counts = []
mega_failed_pairs = []
mega_failure_reasons = {}

global_idx = 0
matcher_outdoor.eval()
for scene_name, ds in zip(scenes, datasets):
    for local_idx in tqdm(range(len(ds)), desc=scene_name):
        pair_pose_error = np.inf
        pair_rot_error = np.inf
        pair_trans_error = np.inf
        pair_matches = 0
        pair_inliers = 0
        failure_reason = None
        try:
            sample = ds[local_idx]
            batch = {
                "image0": sample["image0"][None].to(DEVICE),
                "image1": sample["image1"][None].to(DEVICE),
                "scale0": sample["scale0"][None].to(DEVICE),
                "scale1": sample["scale1"][None].to(DEVICE),
            }
            if "mask0" in sample:
                batch["mask0"] = sample["mask0"][None].to(DEVICE)
                batch["mask1"] = sample["mask1"][None].to(DEVICE)
            with torch.no_grad():
                matcher_outdoor(batch)
            pts0 = batch["mkpts0_f"].detach().cpu().numpy()
            pts1 = batch["mkpts1_f"].detach().cpu().numpy()
            pair_matches = len(pts0)
            K0, K1 = sample["K0"].numpy(), sample["K1"].numpy()
            T_gt = sample["T_0to1"].numpy()
            ret = estimate_pose_repo(pts0, pts1, K0, K1, thresh=0.5, conf=0.99999)
            if ret is None:
                failure_reason = "pose estimation returned None"
            else:
                R, t, inliers = ret
                pair_inliers = int(inliers.sum())
                pair_trans_error, pair_rot_error = relative_pose_error_repo(T_gt, R, t)
                pair_pose_error = max(pair_trans_error, pair_rot_error)
        except Exception as exc:
            failure_reason = f"{type(exc).__name__}: {exc}"

        mega_pose_errors.append(float(pair_pose_error))
        mega_rotation_errors.append(float(pair_rot_error))
        mega_translation_errors.append(float(pair_trans_error))
        mega_match_counts.append(int(pair_matches))
        mega_inlier_counts.append(int(pair_inliers))
        if not np.isfinite(pair_pose_error):
            mega_failed_pairs.append(global_idx)
            mega_failure_reasons[str(global_idx)] = failure_reason or "non-finite pose error"
        global_idx += 1

        # Save a recovery checkpoint after every 100 pairs; not used for official metrics.
        if global_idx % 100 == 0:
            np.save(RESULTS_DIR / "megadepth_pose_errors_partial.npy", np.asarray(mega_pose_errors, dtype=float))

assert len(mega_pose_errors) == len(mega_match_counts) == len(mega_inlier_counts) == 1500
print("MegaDepth pairs:", len(mega_pose_errors), "failures:", len(mega_failed_pairs))
if mega_failed_pairs:
    print("First failure:", mega_failed_pairs[0], mega_failure_reasons[str(mega_failed_pairs[0])])


In [ ]:
mega_aucs = error_auc(mega_pose_errors)
finite_mega = np.asarray(mega_pose_errors, dtype=float)
finite_mega = finite_mega[np.isfinite(finite_mega)]
if len(finite_mega) == 0:
    raise RuntimeError("MegaDepth produced no valid camera poses; check data and checkpoint settings.")
mega_results = {
    "dataset": "MegaDepth-1500",
    "checkpoint": OUTDOOR_CKPT.name,
    "num_pairs": len(mega_pose_errors),
    "failed_pairs": len(mega_failed_pairs),
    "auc_5": float(mega_aucs["AUC@5°"]),
    "auc_10": float(mega_aucs["AUC@10°"]),
    "auc_20": float(mega_aucs["AUC@20°"]),
    "median_pose_error_deg": float(np.median(finite_mega)),
    "average_matches": float(np.mean(mega_match_counts)),
    "average_ransac_inliers": float(np.mean(mega_inlier_counts)),
}
(RESULTS_DIR / "megadepth_reproduction_results.json").write_text(json.dumps(mega_results, indent=2))
(RESULTS_DIR / "megadepth_failure_reasons.json").write_text(json.dumps(mega_failure_reasons, indent=2))
np.save(RESULTS_DIR / "megadepth_pose_errors_final.npy", np.asarray(mega_pose_errors, dtype=float))
print("MegaDepth AUC (%)", {k: round(v * 100, 2) for k, v in mega_aucs.items()})
print("MegaDepth median pose error (degrees):", mega_results["median_pose_error_deg"])
print("Saved MegaDepth results under", RESULTS_DIR)


## 4. Results summary (computed from the current execution)

In [ ]:
import pandas as pd
summary = pd.DataFrame([
    {"Dataset": r["dataset"], "Pairs": r["num_pairs"], "Failed": r["failed_pairs"],
     "AUC@5° (%)": r["auc_5"]*100, "AUC@10° (%)": r["auc_10"]*100,
     "AUC@20° (%)": r["auc_20"]*100,
     "Median error (°)": r["median_pose_error_deg"]}
    for r in (scannet_results, mega_results)
])
print(summary.round(2).to_string(index=False))
summary.to_csv(RESULTS_DIR / "loftr_reproduction_summary.csv", index=False)
print("Saved:", RESULTS_DIR / "loftr_reproduction_summary.csv")


## 5. Reference results and attribution

### Results recorded during development

The values below were recorded in the earlier evaluation notebook (`LoFTR_SETH.ipynb`) and reported in the group presentation. They are provided as **previously obtained reference measurements**, not as outputs of a fresh execution of this revised notebook. The current notebook calculates and saves its own results in Section 4 when executed with the required datasets and checkpoints.

| Benchmark | AUC@5° | AUC@10° | AUC@20° | Failed pose estimates |
|---|---:|---:|---:|---:|
| ScanNet-1500 | 21.52% | 40.37% | 57.18% | 74 / 1,500 |
| MegaDepth-1500 | 52.80% | 69.19% | 81.19% | 9 / 1,500 |

The original debugging experiment on 50 MegaDepth pairs identified missing coordinate-scaling inputs and a checkpoint-configuration mismatch; these corrections are included in the current evaluation code. The revised notebook has been structurally checked but has **not** been fully re-executed against the external benchmark data and model weights in this workspace. Numerical agreement with the earlier results therefore remains to be confirmed by execution.

### Contributions and AI assistance

The reproduction objectives, experimental direction, troubleshooting priorities, interpretation of results, and presentation of findings were guided by the student contributor within the project team. **OpenAI ChatGPT assisted substantially with generating and adapting Python evaluation code, explaining the underlying methods, debugging implementation issues, and organizing the notebook and documentation.** The resulting evaluation combines researcher-directed decisions with AI-assisted implementation and analysis. The LoFTR architecture, original source implementation, benchmark datasets, and pretrained checkpoints are credited to their respective authors. Responsibility for validating and submitting the final reproduction remains with the project team.


In [2]:
import os

if not os.path.isdir("/content/LoFTR/.git"):
    !git clone https://github.com/matthewpinto1102-web/LoFTR.git /content/LoFTR

%cd /content/LoFTR

Cloning into '/content/LoFTR'...
remote: Enumerating objects: 432, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 432 (delta 133), reused 105 (delta 105), pack-reused 229 (from 2)
Receiving objects: 100% (432/432), 31.75 MiB | 22.88 MiB/s, done.
Resolving deltas: 100% (160/160), done.
/content/LoFTR


In [3]:
%cd /content/LoFTR

# Check the current branch and files
!git branch --show-current
!git status
!git ls-files reproduction/Sethmini/

/content/LoFTR
master
On branch master
Your branch is up to date with 'origin/master'.

nothing to commit, working tree clean


In [6]:
# Replace with your exact new notebook filename
!git add "reproduction/Sethmini/NEW_NOTEBOOK.ipynb"

# Remove the old notebook only if this is its actual tracked path
!git rm "reproduction/Sethmini/LoFTR_SETH_CLEAN_REVIEWED.ipynb"

# Review changes before committing
!git diff --cached --stat

fatal: pathspec 'reproduction/Sethmini/NEW_NOTEBOOK.ipynb' did not match any files
fatal: pathspec 'reproduction/Sethmini/LoFTR_SETH_CLEAN_REVIEWED.ipynb' did not match any files


In [5]:
%cd /content/LoFTR

# Find all notebooks tracked by Git
!git ls-files "*.ipynb"

# Check existing repository folders
!ls -la

/content/LoFTR
notebooks/demo_single_pair.ipynb
notebooks/visualize_dump_results.ipynb
total 100
drwxr-xr-x 12 root root  4096 Nov 13 23:49 .
drwxr-xr-x  1 root root  4096 Nov 13 23:49 ..
drwxr-xr-x  6 root root  4096 Nov 13 23:49 assets
drwxr-xr-x  4 root root  4096 Nov 13 23:49 configs
drwxr-xr-x  4 root root  4096 Nov 13 23:49 data
drwxr-xr-x  2 root root  4096 Nov 13 23:49 demo
drwxr-xr-x  2 root root  4096 Nov 13 23:49 docs
-rw-r--r--  1 root root   238 Nov 13 23:49 environment.yaml
drwxr-xr-x  8 root root  4096 Nov 13 23:50 .git
-rw-r--r--  1 root root   370 Nov 13 23:49 .gitignore
-rw-r--r--  1 root root   167 Nov 13 23:49 .gitmodules
-rw-r--r--  1 root root 11358 Nov 13 23:49 LICENSE
drwxr-xr-x  2 root root  4096 Nov 13 23:49 notebooks
-rw-r--r--  1 root root  8532 Nov 13 23:49 README.md
-rw-r--r--  1 root root   370 Nov 13 23:49 requirements.txt
drwxr-xr-x  4 root root  4096 Nov 13 23:49 scripts
drwxr-xr-x  9 root root  4096 Nov 13 23:49 src
-rw-r--r--  1 root root  2628 Nov 1

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

# Upload the downloaded reproduction notebook
uploaded = files.upload()

# Copy it into the team's notebook directory
if len(uploaded) == 1:
    filename = next(iter(uploaded))
    destination = Path("/content/LoFTR/notebooks") / filename

    shutil.copy2(filename, destination)
    print("Notebook copied to:", destination)
else:
    print("Please upload only one notebook.")